<a href="https://colab.research.google.com/github/RvXp/Topicos-Especiais-em-IA-LLM/blob/main/Transformer_Decoder_only_and_Inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 3 - Transformer *decoder-only*

Nesta aula você irá modificar o Transformer *decoder-only* fornecido a seguir.
Observe que em um *decoder-only* não existe:

*   cross-attention
*   encoder separado

e é utilizado com *auto-regressão*.

## Objetivo


## Exercício

Neste exercício você deve:

1.   carregar o seu conjunto de documentos
2.   treinar e usar (ou carregar) um tokenizador
3.   fazer treino de um modelo decoder-only
4.   incluir no loop de treino, inferência usando máxima probabilidade
5.   incluir no loop de treino, inferência usando amostragem com temperatura


In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F

# Passo 1: carregar conjunto de documentos

In [4]:
from google.colab import drive
import os
import re

# 1. Montar o Google Drive para acessar a pasta "IA - LLM"
drive.mount('/content/drive')

# 2. Caminho para o seu arquivo frases.txt
# No Google Drive, o caminho padrão começa com '/content/drive/My Drive/'
path_frases = '/content/drive/My Drive/IA - LLM/frases.txt'

def clean_ascii(text):
    """Limpa caracteres não-ascii e símbolos conforme o código original"""
    text = text.encode("ascii", errors="ignore").decode()
    return re.sub(r"[^A-Za-z0-9 .,:;!?'\-]", "", text)

# 3. Carregar e processar os documentos
documentos = []

if os.path.exists(path_frases):
    with open(path_frases, 'r', encoding='utf-8') as f:
        for linha in f:
            # Limpa a linha (strip remove \n) e aplica a limpeza ascii
            texto = clean_ascii(linha.strip())

            # Opcional: manter a lógica de split original se suas frases tiverem datas/anos
            texto = texto.split(" - ")[0]

            if len(texto) > 0:
                documentos.append(texto)

    print("Total documentos carregados do Drive:", len(documentos))
    print("Sample:", documentos[:10])
else:
    print(f"Erro: O arquivo não foi encontrado no caminho: {path_frases}")
    print("Verifique se o nome da pasta 'IA - LLM' ou do arquivo está correto no seu Drive.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Total documentos carregados do Drive: 753
Sample: ['O homem  a medida de todas as coisas, das coisas que so, enquanto so, das coisas que no so, enquanto no so.', 'S sei que nada sei.', 'A vida no examinada no vale a pena ser vivida.', 'Ningum entra no mesmo rio duas vezes, pois tudo flui e nada permanece.', 'O ser  e o no-ser no .', 'A virtude  o nico bem; o vcio, o nico mal.', 'Onde h amor e sabedoria, no h medo nem ignorncia.', 'A felicidade depende de ns mesmos.', 'O homem  um animal poltico.', 'A coragem  a primeira das qualidades humanas, porque garante todas as outras.']


# Passo 2: Carregar ou treinar um tokenizador

In [6]:
from tokenizers import Tokenizer
from tokenizers.models import WordLevel
from tokenizers.trainers import WordLevelTrainer
from tokenizers.pre_tokenizers import Whitespace
import torch # Certifique-se de importar o torch aqui
import random

# 1. Configuração do Tokenizador em nível de palavra
# O unk_token "[UNK]" lida com palavras que não aparecerem no treino
tokenizer = Tokenizer(WordLevel(unk_token="[UNK]"))
tokenizer.pre_tokenizer = Whitespace()

# 2. Configuração do Trainer
# Definimos o tamanho do vocabulário e os tokens especiais
trainer = WordLevelTrainer(
    vocab_size=5000, # Aumentei um pouco para cobrir mais variações do seu txt
    special_tokens=["[PAD]", "[UNK]", "[BOS]", "[EOS]"]
)

# 3. Treino do tokenizador usando a lista 'documentos' que carregamos do seu Drive
tokenizer.train_from_iterator(documentos, trainer)

# 4. Funções auxiliares ajustadas
def encode(text):
    # Adiciona manualmente o início (BOS) e fim (EOS) da sequência
    ids = tokenizer.encode("[BOS] " + text + " [EOS]").ids
    return torch.tensor(ids, dtype=torch.long)

def decode(ids):
    # Converte os IDs de volta para texto legível
    if isinstance(ids, torch.Tensor):
        ids = ids.tolist()
    return tokenizer.decode(ids)

# Define o tamanho final do vocabulário para o modelo Transformer
vocab_size = tokenizer.get_vocab_size()

print(f"Vocabulário treinado com sucesso! Tamanho: {vocab_size}")

Vocabulário treinado com sucesso! Tamanho: 1706


## Definição do modelo Transformer Decoder-only

In [7]:
class DecoderOnlyTransformer(nn.Module):
    "Implementação de um transformer que tem somente a parte do decoder"
    def __init__(self, vocab_size, d_model=128, n_heads=4, num_layers=3, max_len=64):
        super().__init__()
        self.max_len = max_len

        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)

        layer = nn.TransformerDecoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=256,
            batch_first=True
        )
        self.decoder = nn.TransformerDecoder(layer, num_layers=num_layers)
        self.lm_head = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        B, T = x.shape
        pos = torch.arange(T, device=x.device).unsqueeze(0)

        h = self.token_emb(x) + self.pos_emb(pos)

        # Máscara causal
        mask = torch.triu(torch.ones(T, T, device=x.device), diagonal=1).bool()

        out = self.decoder(h, h, tgt_mask=mask)
        logits = self.lm_head(out)
        return logits

# Códigos de inferência

## Código de inferência simples: token com maior probabilidade


In [8]:
def max_prob_sampling(logits):
    next_token = logits.argmax(dim=-1)
    return next_token.unsqueeze(0)

## Código de inferência avançada: amostragem com temperatura

In [19]:
import torch
import torch.nn.functional as F
import random

# Função para Máxima Probabilidade (Greedy)
def max_prob_sampling(logits):
    next_token = logits.argmax(dim=-1)
    return next_token.unsqueeze(0)

# Função para Amostragem (Temperature/Top-P)
def sampling(logits, top_p=0.9, top_k=None, temperature=1.0):
    logits = logits / max(temperature, 1e-5)

    if top_k is not None:
        k_to_use = min(top_k, logits.size(-1))
        v, _ = torch.topk(logits, k_to_use)
        logits[logits < v[:, [-1]]] = float('-inf')

    sorted_logits, sorted_indices = torch.sort(logits, descending=True)
    cumulative_probs = torch.softmax(sorted_logits, dim=-1).cumsum(dim=-1)

    mask = cumulative_probs > top_p
    mask[..., 1:] = mask[..., :-1].clone()
    mask[..., 0] = False

    filtered_logits = sorted_logits.masked_fill(mask, float('-inf'))
    probs = torch.softmax(filtered_logits, dim=-1)
    sampled_idx = torch.multinomial(probs, num_samples=1)
    return sorted_indices[sampled_idx]

# Função de Geração Unificada
def generate(prompt, next_token_function, max_new_tokens=20, top_k=None, top_p=0.9, temperature=1.0):
    model.eval() # Modo de avaliação
    x = encode(prompt).unsqueeze(0).to(device)

    with torch.no_grad():
        for _ in range(max_new_tokens):
            logits = model(x)[:, -1, :]

            if next_token_function == sampling:
                next_token = next_token_function(logits.squeeze(0), top_k=top_k, top_p=top_p, temperature=temperature)
            else:
                next_token = next_token_function(logits.squeeze(0))

            x = torch.cat([x, next_token.unsqueeze(0)], dim=1)

            if next_token.item() == tokenizer.token_to_id("[EOS]"):
                break
    return decode(x[0])

# 4. Fazer treino de modelo: códigos de treino

In [20]:
# Função auxiliar para gerar batches de exemplos para treino
def sample_batch(batch_size=16, max_len=20):
    batch = random.sample(documentos, batch_size)
    tokenized = [encode(t) for t in batch]

    max_t = min(max(len(x) for x in tokenized), max_len)
    padded = []

    for x in tokenized:
        x = x[:max_t]
        pad_len = max_t - len(x)
        if pad_len > 0:
            x = torch.cat([x, torch.zeros(pad_len, dtype=torch.long)])
        padded.append(x)

    return torch.stack(padded)

# Configurações iniciais
device = "cuda" if torch.cuda.is_available() else "cpu"
model = DecoderOnlyTransformer(vocab_size).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)
steps = 10000
prompt_teste = "O sistema é" # Prompt baseado no seu contexto de segurança/CZAR

for step in range(1, steps + 1):
    model.train() # Garante modo de treino
    batch = sample_batch().to(device)

    # Cálculo da Loss
    logits = model(batch[:, :-1])
    loss = F.cross_entropy(
        logits.reshape(-1, vocab_size),
        batch[:, 1:].reshape(-1),
        ignore_index=tokenizer.token_to_id("[PAD]")
    )

    # Otimização
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # Logs de Progresso
    if step % 50 == 0:
        ppl = torch.exp(loss).item()
        print(f"[step {step}] loss={loss.item():.4f}, ppl={ppl:.2f}")

    # Inferência Combinada
    if step % 100 == 0:
        print(f"\n--- Validação no Step {step} ---")

        # 1. Máxima Probabilidade (Gera o texto mais "seguro" e provável)
        out_max = generate(prompt_teste, next_token_function=max_prob_sampling)
        print(f"MAXPROB: {out_max}")

        # 2. Amostragem com Temperatura (Gera textos com variação)
        out_sample = generate(prompt_teste, next_token_function=sampling, temperature=0.8)
        print(f"SAMPLING (T=0.8): {out_sample}")

        print("-" * 40)

print("Treinamento finalizado.")

[step 50] loss=7.4245, ppl=1676.55
[step 100] loss=7.2155, ppl=1360.39

--- Validação no Step 100 ---
MAXPROB: O sistema antecipar No todos comunidade baralha ideologia baralha pensa sorte guerra descansar comunidade conceitos qualidade universais sorte guerra sentido critrios resultar
SAMPLING (T=0.8): O sistema crtica conceitos vcio isso resistncia crtica carne infantil existisse raro ignorantes representao negativo mnadas supomos facilmente muito versa inquieto leitura
----------------------------------------
[step 150] loss=6.9398, ppl=1032.61
[step 200] loss=6.9415, ppl=1034.33

--- Validação no Step 200 ---
MAXPROB: O sistema Viver ordem civilizao princpio .
SAMPLING (T=0.8): O sistema poeira organizada das . suave interpretaes soma poltica indivduo lanterna fabricar segundo orientalismo ouvimos negativo Tu sequer perdeu escreve
----------------------------------------
[step 250] loss=6.5980, ppl=733.61
[step 300] loss=6.4482, ppl=631.59

--- Validação no Step 300 ---
MAXPROB: O 

incluir no loop de treino, inferência usando máxima probabilidade

# Exercício: controle de temperatura, Top-K e Top-P

Modifique o código a seguir para fazer a visualização de produção de tokens do modelo que você treinou.

In [22]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, Markdown

# 1. Função para capturar os logits reais do modelo para um prompt
def get_model_logits(prompt):
    model.eval()
    with torch.no_grad():
        # Transforma o texto em tensor e move para a GPU/CPU
        x = encode(prompt).unsqueeze(0).to(device)
        # Obtém a saída do modelo
        logits = model(x)
        # Pegamos apenas os logits do ÚLTIMO token da sequência
        last_token_logits = logits[0, -1, :]
    return last_token_logits

# 2. Widgets de Interface
# Criamos um campo de texto para você digitar qualquer prompt do seu 'frases.txt'
prompt_input = widgets.Text(
    value='O sistema é',
    placeholder='Digite o início de uma frase...',
    description='Prompt:',
    layout=widgets.Layout(width='95%')
)

temp_slider = widgets.FloatSlider(value=1.0, min=0.1, max=5.0, step=0.1, description="Temperatura")
topk_widget = widgets.IntSlider(value=10, min=1, max=20, step=1, description="Top-K")
topp_widget = widgets.FloatSlider(value=1.0, min=0.1, max=1.0, step=0.05, description="Top-P")

output_plot = widgets.Output()

# 3. Lógica de atualização baseada no modelo treinado
def update_real_plot(*args):
    with output_plot:
        output_plot.clear_output(wait=True)

        # Obtém os logits reais do seu modelo
        raw_logits = get_model_logits(prompt_input.value)

        # Aplica a Temperatura
        logits = raw_logits / max(temp_slider.value, 1e-5)

        # Filtro Top-K
        v, _ = torch.topk(logits, min(topk_widget.value, vocab_size))
        logits_k = logits.clone()
        logits_k[logits < v[-1]] = float('-inf')

        # Softmax para obter probabilidades
        probs = torch.softmax(logits_k, dim=-1)

        # Filtro Top-P (Nucleus)
        sorted_probs, sorted_indices = torch.sort(probs, descending=True)
        cumulative_probs = torch.cumsum(sorted_probs, dim=-1)

        # Máscara para Top-P
        mask = cumulative_probs > topp_widget.value
        mask[1:] = mask[:-1].clone()
        mask[0] = False
        sorted_probs[mask] = 0

        # Normaliza novamente as probabilidades após os cortes
        final_probs = sorted_probs / sorted_probs.sum()

        # Pegamos os top 10 tokens finais para exibir no gráfico
        top_n = 10
        display_probs = final_probs[:top_n].cpu().numpy()
        display_indices = sorted_indices[:top_n].cpu().numpy()
        # Converte IDs de volta para palavras usando seu decoder
        display_tokens = [tokenizer.id_to_token(int(i)) for i in display_indices]

        # Renderização do Gráfico
        fig, ax = plt.subplots(figsize=(12, 5))
        bars = ax.bar(display_tokens, display_probs * 100, color='#3498db')
        ax.set_ylim(0, 105)
        ax.set_ylabel("Probabilidade (%)")
        ax.set_title(f"Previsão do Próximo Token para: '{prompt_input.value}'")
        plt.xticks(rotation=45)

        for bar, p in zip(bars, display_probs):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                    f"{p*100:.1f}%", ha='center', fontsize=9)
        plt.show()

# Conecta os widgets à função de atualização
for w in [prompt_input, temp_slider, topk_widget, topp_widget]:
    w.observe(update_real_plot, names="value")

# Exibição do Layout
display(Markdown("### Visualização de Inferência com Modelo Real"))
display(widgets.VBox([
    prompt_input,
    widgets.HBox([temp_slider, topk_widget, topp_widget]),
    output_plot
]))

update_real_plot()

### Visualização de Inferência com Modelo Real

In [ ]:
# @title
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, Markdown

# -----------------------------
# Mock language-model logits
# -----------------------------
TOKENS = ["blue", "purple", "violet", "vio", "not", "Blue", "green", "gray", "grey", "black"]
BASE_LOGITS = np.array([6.0, 1.5, 0.5, 0.2, 0.2, 0.0, -5.0, -5.0, -5.0, -5.0])

def softmax(x):
    e = np.exp(x - np.max(x))
    return e / e.sum()

def apply_temperature(logits, temperature):
    return logits / max(temperature, 1e-5)

def apply_top_k(probs, k):
    if k >= len(probs):
        return probs
    idx = np.argsort(probs)[::-1]
    mask = np.zeros_like(probs)
    mask[idx[:k]] = 1
    probs = probs * mask
    return probs / probs.sum()

def apply_top_p(probs, p):
    idx = np.argsort(probs)[::-1]
    cumulative = np.cumsum(probs[idx])
    mask = cumulative <= p
    mask[np.argmax(mask)] = True
    new_probs = np.zeros_like(probs)
    new_probs[idx[mask]] = probs[idx[mask]]
    return new_probs / new_probs.sum()

# -----------------------------
# Widgets
# -----------------------------
prompt_dropdown = widgets.Dropdown(
    options=["Roses are red, violets are..."],
    value="Roses are red, violets are...",
    description="Prompt",
    layout=widgets.Layout(width="95%")
)

temperature_slider = widgets.FloatSlider(
    value=1.0, min=0.1, max=5.0, step=0.1,
    description="Temperatura",
    readout_format=".1f",
    layout=widgets.Layout(width="90%")
)

topk_slider = widgets.IntSlider(
    value=6, min=1, max=10, step=1,
    description="Top-K",
    layout=widgets.Layout(width="90%")
)

topp_slider = widgets.FloatSlider(
    value=1.0, min=0.1, max=1.0, step=0.05,
    description="Top-P",
    readout_format=".2f",
    layout=widgets.Layout(width="90%")
)

# -----------------------------
# Plot
# -----------------------------
output_plot = widgets.Output()

def update_plot(*args):
    with output_plot:
        output_plot.clear_output()

        logits = apply_temperature(BASE_LOGITS, temperature_slider.value)
        probs = softmax(logits)
        probs = apply_top_k(probs, topk_slider.value)
        probs = apply_top_p(probs, topp_slider.value)

        fig, ax = plt.subplots(figsize=(10, 4))
        bars = ax.bar(TOKENS, probs * 100)
        ax.set_ylim(0, 100)
        ax.set_ylabel("Probabilidade (%)")
        ax.set_title("Probabilidade do próximo token")

        for bar, p in zip(bars, probs):
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height(),
                f"{p*100:.2f}%",
                ha="center",
                va="bottom",
                fontsize=9
            )

        plt.xticks(rotation=0)
        plt.show()

for w in [temperature_slider, topk_slider, topp_slider]:
    w.observe(update_plot, names="value")

# -----------------------------
# Collapsible explanation
# -----------------------------
accordion = widgets.Accordion(
    children=[widgets.HTML(
        """
        <ul>
          <li><b>Temperatura</b>: Controla aleatoriedade com mudança na escala dos logits.</li>
          <li><b>Top-K</b>: Restringe a amostra aos K tokens mais prováveis.</li>
          <li><b>Top-P</b>: Usa somente os menor conjunto de tokens que resultam em probabilidade acumulada até P.</li>
        </ul>
        """
    )]
)
accordion.set_title(0, "Entenda a visualização")

# -----------------------------
# Layout
# -----------------------------
controls = widgets.VBox([
    prompt_dropdown,
    widgets.HBox([temperature_slider]),
    widgets.HBox([topk_slider]),
    widgets.HBox([topp_slider])
])

display(Markdown("## Visualização de controle de Temperatura, Top-K e Top-P."))
display(widgets.VBox([
    widgets.HTML("<b>Parameters</b>"),
    controls,
    output_plot,
    accordion
]))

update_plot()


## Visualização de controle de Temperatura, Top-K e Top-P.